### 任务规划中间件 TodoListMiddleware

`TodoListMiddleware` 让 Agent 具备**任务规划**能力：它会自动给 Agent 注入一个 `write_todos` 工具和一段使用说明，让 Agent 在处理多步骤任务时先列出待办清单，并在执行过程中实时更新每一项的状态。

它主要做了三件事：

1. **注入 `write_todos` 工具**：参数是一个 `Todo` 列表
2. **新增 `todos` 状态**：每次调用都会把整个列表写入 Agent state 的 `todos` 字段
3. **注入系统提示 + 并行调用保护**：指导何时使用，并禁止同一轮并行多次调用

#### 构造参数

```python
TodoListMiddleware(
    *,
    system_prompt=WRITE_TODOS_SYSTEM_PROMPT,
    tool_description=WRITE_TODOS_TOOL_DESCRIPTION,
)
```

- `system_prompt`：追加到系统消息末尾的提示词，用来指导 Agent **何时、如何**使用 `write_todos`（默认是一段很详细的说明，强调只在复杂多步任务时才用）
- `tool_description`：`write_todos` **工具本身的描述**，模型靠它决定是否调用（默认同样是一段详细说明）

> 两个参数都是关键字参数；不传则使用内置默认值。

#### 内置工具与状态

工具签名：`write_todos(todos: list[Todo]) -> Command`，其中 `Todo` 结构为：

| 字段 | 类型 | 说明 |
| --- | --- | --- |
| `content` | `str` | 任务内容/描述 |
| `status` | `"pending" \| "in_progress" \| "completed"` | 任务状态 |

- 每次调用会**整体替换** `todos` 列表（不是追加），所以一次可以同时更新多项
- 结果在 Agent state 的 `todos` 键中读取
- `after_model` 会拦截「同一轮并行多次调用 `write_todos`」，为每次调用返回一条 `status="error"` 的 `ToolMessage`，避免列表更新顺序冲突

In [1]:
import json
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

model = init_chat_model(
    api_base=os.getenv("DEEPSEEK_API_BASE"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    model="deepseek-flash",
    model_provider="deepseek",
    model_kwargs={"reasoning_effort": "none"},
)

# 查看中间件内置的 write_todos 工具及其入参 schema
middleware = TodoListMiddleware()
print("内置工具：", [t.name for t in middleware.tools])
print("write_todos 入参 schema：")
print(json.dumps(middleware.tools[0].args_schema.model_json_schema(), ensure_ascii=False, indent=2))


/Users/lijixu/PycharmProjects/langchain_demo/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


内置工具： ['write_todos']
write_todos 入参 schema：
{
  "$defs": {
    "Todo": {
      "description": "A single todo item with content and status.",
      "properties": {
        "content": {
          "title": "Content",
          "type": "string"
        },
        "status": {
          "enum": [
            "pending",
            "in_progress",
            "completed"
          ],
          "title": "Status",
          "type": "string"
        }
      },
      "required": [
        "content",
        "status"
      ],
      "title": "Todo",
      "type": "object"
    }
  },
  "description": "Input schema for the `write_todos` tool.",
  "properties": {
    "todos": {
      "items": {
        "$ref": "#/$defs/Todo"
      },
      "title": "Todos",
      "type": "array"
    }
  },
  "required": [
    "todos"
  ],
  "title": "WriteTodosInput",
  "type": "object"
}


/Users/lijixu/PycharmProjects/langchain_demo/.venv/lib/python3.14/site-packages/langchain/chat_models/base.py:516: UserWarning: Parameters {'reasoning_effort'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return _init_chat_model_helper(


#### 案例 1：默认用法——让 Agent 规划多步任务

In [ ]:
# 只要挂上中间件即可，无需额外传 tools；Agent 会自己决定是否调用 write_todos
agent = create_agent(
    model=model,
    middleware=[TodoListMiddleware()],
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "帮我规划一个三天两夜的北京旅游行程，请先列出待办事项，再逐步完成。",
            }
        ]
    }
)

# 最终待办列表就在 state 的 todos 字段里
print("最终 todos：")
for item in result["todos"]:
    print(f"  [{item['status']}] {item['content']}")


#### 案例 2：观察状态流转（多次 write_todos 调用）

In [ ]:
# write_todos 会被多次调用：先建列表（第一项 in_progress），完成一项就更新一次
# 下面的循环打印出每一次调用时的状态快照，可以看到任务的推进过程
call_index = 1
for message in result["messages"]:
    for tool_call in getattr(message, "tool_calls", None) or []:
        if tool_call["name"] != "write_todos":
            continue
        print(f"---- 第 {call_index} 次 write_todos ----")
        for todo in tool_call["args"]["todos"]:
            print(f"  [{todo['status']}] {todo['content']}")
        call_index += 1


#### 案例 3：自定义 system_prompt 与 tool_description

In [ ]:
# 通过自定义这两个参数，可以改变 Agent 使用 todo 的倾向和工具描述
custom_middleware = TodoListMiddleware(
    system_prompt="你是一个任务规划助手，遇到复杂任务时请始终使用 write_todos 来规划步骤。",
    tool_description="创建或更新当前任务的待办事项列表。",
)

agent_custom = create_agent(model=model, middleware=[custom_middleware])

result_custom = agent_custom.invoke(
    {"messages": [{"role": "user", "content": "帮我规划学习 Python 的路线，分步骤进行"}]}
)

print("自定义提示后得到的 todos：")
for item in result_custom["todos"]:
    print(f"  [{item['status']}] {item['content']}")


#### 案例 4：并行调用保护（确定性演示，不调用模型）

In [ ]:
# 若同一轮里模型并行调用了两次 write_todos，中间件会在 after_model 阶段拦截，
# 为每个调用返回一条 status="error" 的 ToolMessage（避免"整体替换"造成顺序冲突）
from langchain_core.messages import AIMessage

fake_ai = AIMessage(
    content="",
    tool_calls=[
        {"name": "write_todos", "args": {"todos": [{"content": "任务A", "status": "pending"}]}, "id": "call_1"},
        {"name": "write_todos", "args": {"todos": [{"content": "任务B", "status": "pending"}]}, "id": "call_2"},
    ],
)

# 直接调用 after_model 观察拦截结果（runtime 在此场景未使用，传 None 即可）
guard_result = middleware.after_model({"messages": [fake_ai]}, None)

print("拦截返回的 ToolMessage 数量：", len(guard_result["messages"]))
for msg in guard_result["messages"]:
    print(f"  status = {msg.status} | {msg.content}")


#### 要点回顾

1. 挂上 `TodoListMiddleware` 后，Agent 自动获得 `write_todos` 工具与 `todos` 状态。
2. `write_todos` 每次会**整体替换**待办列表，因此一次调用可同时更新多项。
3. `Todo` 由 `content` + `status`(`pending`/`in_progress`/`completed`) 组成。
4. `system_prompt` 控制「是否/如何使用」，`tool_description` 控制工具描述，两者都可用默认值或自定义。
5. 同一轮并行多次调用 `write_todos` 会被拦截并返回错误消息。
6. 该中间件适合多步骤复杂任务；简单任务模型通常会直接完成而不调用它。